# Stage 2b — W2V2-L2 Embedding Extraction
### Alertreck · Transfer Learning Pre-processing

This notebook extracts frozen **wav2vec 2.0 layer-2** embeddings from raw audio and saves them as `.npz` shards for use in `04a-train-w2v2-l2.ipynb`.

The encoder (`facebook/wav2vec2-base`) is **fully frozen** and run once. Training the linear classification head never touches the encoder again, so GPU is only needed here.

---

## Pipeline summary

```
Raw audio  →  resample 16 kHz  →  3-second windows (50% hop)
          →  frozen W2V2 encoder  →  hidden_states[2]  →  mean-pool
          →  L2-normalise  →  768-dim embedding  →  .npz shard
```

## Output layout

```
data/processed/w2v2_l2/
  train/          shard_NNN.npz   X: (N, 768) float32   y: (N,) int64
  val/            shard_NNN.npz
  test/           shard_NNN.npz
  train_aug_A/    shard_NNN.npz
  train_aug_B/    shard_NNN.npz
  train_aug_C/    shard_NNN.npz
  manifest.json
```

## Requirements
- GPU accelerator (T4 or P100 recommended)
- Datasets attached: **raw audio** + **alertreck-mel2** (for `splits.json`)

## Cell 1 — Install dependencies

In [ ]:
!pip install -q transformers torch torchaudio librosa soundfile tqdm

## Cell 2 — Clone repository

In [ ]:
import os
from pathlib import Path

REPO = Path("/kaggle/working/alertreck")

if not REPO.exists():
    !git clone https://github.com/mangaorphy/alertreck.git {REPO}
else:
    print(f"Repo already exists at {REPO}")

print("Repo contents:", list(REPO.iterdir()))

## Cell 3 — Locate datasets and set up paths

Run this cell to find where your Kaggle input datasets are mounted.

In [ ]:
# Find splits.json (from the mel dataset)
print("=== Looking for splits.json ===")
!find /kaggle/input -name "splits.json" 2>/dev/null

print("\n=== Kaggle input datasets ===")
!ls /kaggle/input/

## Cell 4 — Configure paths

Update `MEL_ROOT` and `AUDIO_ROOT` based on what you saw in Cell 3.

In [ ]:
import shutil

# ── UPDATE THESE TWO PATHS ──────────────────────────────────────────────────
MEL_ROOT   = Path("/kaggle/input/alertreck-mel2/mel")   # folder containing splits.json
AUDIO_ROOT = Path("/kaggle/input/YOUR-AUDIO-DATASET")   # root of your raw audio dataset
# ───────────────────────────────────────────────────────────────────────────

# Copy splits.json to the location the script expects
splits_src  = MEL_ROOT / "splits.json"
splits_dst  = REPO / "data/processed/splits.json"
splits_dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(splits_src, splits_dst)
print(f"Copied splits.json → {splits_dst}")

# Symlink audio dataset so the noise pool loader finds background_wind_rain/
# Adjust the sub-path if your audio files sit inside a sub-folder
audio_src    = AUDIO_ROOT / "dataset"   # e.g. .../dataset/background_animals/...
dataset_link = REPO / "dataset"
if not dataset_link.exists():
    dataset_link.symlink_to(audio_src)
    print(f"Symlinked dataset → {audio_src}")
else:
    print(f"dataset link already exists")

print("\nsplits.json exists :", splits_dst.exists())
print("dataset link exists :", dataset_link.exists())

## Cell 5 — Verify splits.json paths

The file records absolute Kaggle paths from when `audio_preprocessing.py` was first run. If the audio dataset is mounted at a different path now, the cell below will remap them.

In [ ]:
import json

splits = json.loads(splits_dst.read_text())

print("=== Sample paths from splits.json ===")
for split_name, items in splits.items():
    cls, path = items[0]
    exists = Path(path).exists()
    print(f"  [{split_name:<6}] {cls:<25} exists={exists}  {path[:80]}")

# Check if any path is broken
broken = [(cls, p) for items in splits.values() for cls, p in items if not Path(p).exists()]
print(f"\nBroken paths: {len(broken)}")
if broken:
    print("  Example broken:", broken[0][1][:80])
    print("\n>>> Update OLD_PREFIX and NEW_PREFIX in the next cell to remap them.")

## Cell 6 — (Optional) Remap broken paths

Only run this cell if Cell 5 reported broken paths.
Set `OLD_PREFIX` to the prefix shown in the broken path, and `NEW_PREFIX` to where the audio dataset is actually mounted.

In [ ]:
# ── Only edit and run this cell if Cell 5 showed broken paths ──────────────
OLD_PREFIX = "/kaggle/input/OLD-DATASET-NAME"   # prefix from splits.json
NEW_PREFIX = str(AUDIO_ROOT)                     # where the audio actually is
# ───────────────────────────────────────────────────────────────────────────

for split_name in splits:
    splits[split_name] = [
        (cls, p.replace(OLD_PREFIX, NEW_PREFIX))
        for cls, p in splits[split_name]
    ]

splits_dst.write_text(json.dumps(splits, indent=2))
print("Paths remapped and splits.json overwritten.")

# Quick sanity check
sample = splits["train"][:3]
for cls, p in sample:
    print(f"  {cls:<25} exists={Path(p).exists()}  {p[:80]}")

## Cell 7 — Check GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

## Cell 8 — Run extraction

Generates clean splits + all three curriculum augmentation phases.

**Expected runtime on T4 GPU:**
- Clean splits (train + val + test): ~15 min
- Phase A (1 copy per window): ~20 min
- Phase B (2 copies): ~30 min
- Phase C (3 copies): ~40 min
- **Total: ~1.5–2 hours**

If you get a GPU out-of-memory error, reduce `--batch-size` to 32.

In [ ]:
os.chdir(REPO)

!python3 scripts/prepare_w2v2_embeddings.py \
    --aug-phase A B C \
    --device cuda \
    --batch-size 64

## Cell 9 — Verify outputs

In [ ]:
import numpy as np

W2V2_OUT = REPO / "data/processed/w2v2_l2"

print("=== Output directories ===")
for d in sorted(W2V2_OUT.iterdir()):
    if d.is_dir():
        shards = list(d.glob("*.npz"))
        total  = sum(np.load(s)["X"].shape[0] for s in shards)
        print(f"  {d.name:<18}  {len(shards):>3} shards  {total:>7,} embeddings")

print("\n=== Manifest ===")
manifest = json.loads((W2V2_OUT / "manifest.json").read_text())
for k, v in manifest.items():
    if k not in ("splits", "label_map", "script_sha256"):
        print(f"  {k}: {v}")

print("\n=== Single shard spot-check ===")
shard = np.load(W2V2_OUT / "train/shard_000.npz")
print(f"  X shape : {shard['X'].shape}   dtype: {shard['X'].dtype}")
print(f"  y shape : {shard['y'].shape}   dtype: {shard['y'].dtype}")
print(f"  X range : [{shard['X'].min():.4f}, {shard['X'].max():.4f}]")
print(f"  L2 norm (first 5): {np.linalg.norm(shard['X'][:5], axis=1)}")

## Cell 10 — Save as Kaggle dataset

Zip the output folder so you can upload it as a new Kaggle dataset named `alertreck-w2v2-embeddings`.
This dataset will be attached to `04a-train-w2v2-l2.ipynb`.

In [ ]:
ZIP_PATH = Path("/kaggle/working/w2v2_embeddings.zip")

print("Zipping outputs — this may take a few minutes...")
!cd {REPO / 'data/processed'} && zip -r {ZIP_PATH} w2v2_l2/

size_mb = ZIP_PATH.stat().st_size / 1e6
print(f"\nZip created: {ZIP_PATH}  ({size_mb:.1f} MB)")
print("\nNext step: go to Data → Upload New Dataset → upload w2v2_embeddings.zip")
print("Name it: alertreck-w2v2-embeddings")